In [36]:
import awswrangler as wr
import pandas as pd

database_name = table_name = "perturbation_evaluations_5"
# Read the table metadata and data using the Glue catalog
df = wr.s3.read_parquet_table(
    table=table_name,
    database=database_name
)
df_meta = pd.read_csv('s3://jdinvestment/perturbations_5/perturbations.csv')[['sim_id', 'parent_id']].set_index('sim_id')
df1 = df.set_index('sim_id').join(df_meta).reset_index()


In [25]:
df.columns

Index(['sim_id', 'f1_2008', 'f2_2020', 'f3_2022', 'f4_terminal',
       'f5_annual_worst', 'f6_2008_terminal', 'dollar_ret_1p', 'dollar_ret_6p',
       'dollar_ret_13p', 'dollar_ret_26p', 'avg_eps_1q', 'avg_eps_2q',
       'avg_eps_4q', 'avg_eps_8q', 'threshold', 'beta', 'mom_decay',
       'qual_decay', 'macro_weights_0', 'macro_weights_1', 'macro_weights_2',
       'macro_weights_3', 'parent_id'],
      dtype='str')

In [46]:
df1.loc[:, 'drawdown'] = df.loc[:, ['f1_2008', 'f2_2020', 'f3_2022']].sum(axis = 1)
df2 = df1[['parent_id', 'drawdown', 'f4_terminal', 'f5_annual_worst', 'f6_2008_terminal']].groupby('parent_id').agg(lambda x: x.std()/x.mean()).reset_index()\
    .rename(columns = {'drawdown': 'drawdown_cv', 'f4_terminal': 'f4_cv', 'f5_annual_worst': 'f5_cv', 'f6_2008_terminal': 'f6_cv'})
df3 = df1[['parent_id', 'drawdown', 'f4_terminal', 'f5_annual_worst', 'f6_2008_terminal']].groupby('parent_id').agg(lambda x: x.mean()).reset_index()\
    .rename(columns = {'drawdown': 'drawdown_mean', 'f4_terminal': 'f4_mean', 'f5_annual_worst': 'f5_mean', 'f6_2008_terminal': 'f6_mean'})
df4 = df2.set_index('parent_id').join(df3.set_index('parent_id')).reset_index()
df4.loc[:, 'crash'] = (df4.f6_mean/408000)**.2 - 1
df4.loc[:, 'boom'] = (df4.f4_mean/408000)**.2 - 1


In [39]:
df4


,parent_id,drawdown_cv,f4_cv,f5_cv,f6_cv,drawdown_mean,f4_mean,f5_mean,f6_mean
0,16095296,0.277432,0.055541,-0.096854,0.056178,2.376920e+06,8.938702e+05,-0.024738,390547.109453
1,24838343,0.243693,0.053250,-0.112025,0.057090,1.694375e+06,1.247498e+06,-0.037682,448832.904007
2,130287417,0.225245,0.044244,-0.084100,0.044196,2.783122e+05,8.937129e+05,-0.038267,698071.225490
3,139394454,0.428628,0.072396,-0.064754,0.057143,8.515702e+05,1.423540e+06,-0.043929,498000.622612
4,202764069,0.400830,0.063158,-0.052956,0.059762,2.621186e+05,6.460168e+05,-0.039159,712917.044659
...,...,...,...,...,...,...,...,...,...
120,9221654405,0.260496,0.045613,-0.122700,0.043099,6.473041e+05,7.523568e+05,-0.028202,693490.319071
121,9247765605,0.312441,0.036738,-0.071318,0.051681,2.028492e+06,8.710726e+05,-0.024542,407731.438638
122,9479321343,0.406915,0.046686,-0.151682,0.061989,9.947225e+05,1.001885e+06,-0.033675,516998.146294
123,9615205960,0.374918,0.071441,-0.054162,0.048737,7.813297e+05,1.403707e+06,-0.044273,504112.947762


In [6]:
df_parents = pd.read_csv('../analysis/top_ranked.csv')
df_parents = df_parents.loc[df_parents['rank']==0]
df_parents.loc[:, 'drawdown'] = df_parents.loc[:, ['f1_2008', 'f2_2020', 'f3_2022']].sum(axis = 1)

In [47]:
import plotly.express as px

# Create the scatter plot
fig = px.scatter(
    df4, 
    x="crash", 
    y="boom",
    hover_data=df4.columns,  # Displays all columns on mouseover
    title="Mean Drawdown vs. F4 Mean Analysis"
)

# Optional: Improve layout for better readability
fig.update_layout(
    xaxis_title="Drawdown Mean",
    yaxis_title="F4 Mean (Return Period)",
    hovermode="closest"
)

fig.show()

In [49]:
import pandas as pd
import numpy as np

def get_pareto_frontier(df, objective_cols, senses):
    """
    Returns a subset of the dataframe containing only the Pareto optimal rows.
    
    Parameters:
    - df: The input DataFrame (e.g., your Pymoo results)
    - objective_cols: List of column names to evaluate (e.g., ['drawdown', 'return'])
    - senses: List of strings ('min' or 'max') corresponding to each column
    """
    # 1. Normalize all objectives to 'minimize' to simplify logic
    # We multiply 'max' objectives by -1
    data = df[objective_cols].to_numpy(copy=True)
    for i, sense in enumerate(senses):
        if sense == 'max':
            data[:, i] *= -1

    n_rows = data.shape[0]
    is_pareto = np.ones(n_rows, dtype=bool)

    # 2. Vectorized Domination Check
    for i in range(n_rows):
        if is_pareto[i]:
            # A row is dominated if another row is <= in all objectives 
            # AND < in at least one objective.
            # Here we check if row 'i' is dominated by any other row
            is_dominated = np.all(data <= data[i], axis=1) & np.any(data < data[i], axis=1)
            
            # If anyone dominates 'i', 'i' is not Pareto
            if np.any(is_dominated):
                is_pareto[i] = False
            else:
                # Optimization: If 'i' is Pareto, it might dominate others. 
                # Mark those it dominates as not Pareto.
                others_dominated = np.all(data[i] <= data, axis=1) & np.any(data[i] < data, axis=1)
                is_pareto[others_dominated] = False

    return df.iloc[is_pareto].copy()

# --- Example Usage ---
objectives = ['crash', 'boom']
senses = ['max', 'max']
pareto_df = get_pareto_frontier(df4, objectives, senses)
print(f"Found {len(pareto_df)} Pareto optimal solutions.")

Found 13 Pareto optimal solutions.


In [52]:
fig = px.scatter(pareto_df, x = 'crash', y = 'boom')
fig.show()

In [53]:
pareto_df

,parent_id,drawdown_cv,f4_cv,f5_cv,f6_cv,drawdown_mean,f4_mean,f5_mean,f6_mean,crash,boom
9,440800849,0.465682,0.039283,-0.079421,0.052609,5.170906e+05,1.033274e+06,-0.041045,571366.061190,0.069673,0.204235
16,775856594,0.368586,0.055893,-0.052370,0.052905,1.080043e+06,1.487067e+06,-0.046149,457689.091851,0.023251,0.295192
19,1127218185,0.395154,0.079670,-0.038639,0.049869,7.805157e+05,1.256916e+06,-0.041273,529947.925956,0.053694,0.252360
21,1279209563,0.288415,0.054430,-0.085056,0.044046,3.332292e+05,9.087279e+05,-0.032964,768238.059848,0.134925,0.173694
27,1708791521,0.236025,0.052869,-0.072128,0.068775,2.366898e+05,9.081281e+05,-0.035839,799203.599995,0.143930,0.173539
38,2652315412,0.409266,0.059591,-0.062127,0.055542,1.001895e+06,1.457517e+06,-0.044910,459872.712763,0.024225,0.290003
41,2789866219,0.249491,0.053848,-0.121107,0.050236,8.401770e+05,1.088562e+06,-0.030591,566413.414929,0.067812,0.216854
62,4729066849,0.256210,0.059617,-0.095005,0.063105,2.223323e+05,8.353271e+05,-0.034696,816289.289521,0.148780,0.154089
70,5091876646,0.342946,0.059182,-0.052081,0.048913,7.198364e+05,1.435127e+06,-0.046432,506771.794825,0.044312,0.286015
71,5112672398,0.387472,0.048934,-0.099214,0.064971,3.420264e+05,9.491581e+05,-0.033123,709842.996343,0.117122,0.183956


In [56]:
df_stars = pareto_df.set_index('parent_id').join(df_parents.rename(columns = {'sim_id': 'parent_id'}).set_index('parent_id')).reset_index()
df_stars

,parent_id,drawdown_cv,f4_cv,f5_cv,f6_cv,drawdown_mean,f4_mean,f5_mean,f6_mean,crash,...,threshold,beta,mom_decay,qual_decay,macro_weights_0,macro_weights_1,macro_weights_2,macro_weights_3,drawdown,rank
0,440800849,0.465682,0.039283,-0.079421,0.052609,5.170906e+05,1.033274e+06,-0.041045,571366.061190,0.069673,...,-0.923102,14.726582,-0.083764,0.937640,0.075024,-0.263376,-0.096548,0.758178,225788.967935,0
1,775856594,0.368586,0.055893,-0.052370,0.052905,1.080043e+06,1.487067e+06,-0.046149,457689.091851,0.023251,...,-0.027034,14.592669,0.355061,-0.911534,0.069908,0.717331,0.178602,0.996642,329420.133400,0
2,1127218185,0.395154,0.079670,-0.038639,0.049869,7.805157e+05,1.256916e+06,-0.041273,529947.925956,0.053694,...,-0.021408,12.341516,0.353213,-0.780485,0.027202,0.590008,0.366858,0.974911,520378.382700,0
3,1279209563,0.288415,0.054430,-0.085056,0.044046,3.332292e+05,9.087279e+05,-0.032964,768238.059848,0.134925,...,-1.266883,13.597204,0.269597,0.736747,-0.893657,0.263073,0.031216,-0.001888,235230.264200,0
4,1708791521,0.236025,0.052869,-0.072128,0.068775,2.366898e+05,9.081281e+05,-0.035839,799203.599995,0.143930,...,-0.981227,12.413621,0.334458,0.948530,-0.820821,0.332177,-0.015010,0.713205,187454.172800,0
5,2652315412,0.409266,0.059591,-0.062127,0.055542,1.001895e+06,1.457517e+06,-0.044910,459872.712763,0.024225,...,-0.027034,14.592669,0.360428,-0.911534,0.215124,0.703268,0.072065,0.996269,318994.423900,0
6,2789866219,0.249491,0.053848,-0.121107,0.050236,8.401770e+05,1.088562e+06,-0.030591,566413.414929,0.067812,...,-0.238858,14.533128,0.389996,0.921689,-0.820248,0.673487,0.095766,0.996002,557449.931200,0
7,4729066849,0.256210,0.059617,-0.095005,0.063105,2.223323e+05,8.353271e+05,-0.034696,816289.289521,0.148780,...,-0.927269,13.292654,0.336799,0.947319,-0.760038,0.414850,0.018906,0.741177,176463.510900,0
8,5091876646,0.342946,0.059182,-0.052081,0.048913,7.198364e+05,1.435127e+06,-0.046432,506771.794825,0.044312,...,-0.021017,13.460658,0.254390,-0.758937,0.080538,0.609213,0.189502,0.992680,297383.258600,0
9,5112672398,0.387472,0.048934,-0.099214,0.064971,3.420264e+05,9.491581e+05,-0.033123,709842.996343,0.117122,...,-0.595104,13.857197,0.892204,0.881839,-0.800406,0.351256,0.186802,0.140818,264586.984900,0


In [ ]:
import pandas as pd
import numpy as np

def calculate_2d_regime_coordinate(current_date, signal_df, params):
    """
    params (12 total):
    - [0:4]:  Alpha for EMA (smoothing) - one per signal.
    - [4:8]:  Linear weights for Magnitude (RM).
    - [8:12]: Linear weights for Persistence (RP).
    """
    # 1. Isolate 2-year window for structural context
    start_date = current_date - pd.DateOffset(years=2)
    history = signal_df.loc[start_date:current_date]
    
    # 2. Calculate Smoothed State (Magnitude RM)
    # Higher alpha = more reactive; Lower alpha = structural macro view
    alphas = params[0:4]
    smoothed = np.array([
        history[col].ewm(alpha=alphas[i]).mean().iloc[-1] 
        for i, col in enumerate(history.columns)
    ])
    rm = np.dot(smoothed, params[4:8])
    
    # 3. Calculate Stability State (Persistence RP)
    # Ratio of 3-month mean to 2-year mean highlights regime 'age'
    history_3m = history.loc[current_date - pd.DateOffset(months=3):]
    # Handle potential zero division with a small epsilon
    persistence_ratios = history_3m.mean() / (history.mean() + 1e-9)
    rp = np.dot(persistence_ratios.values, params[8:12])
    
    return rm, rp

In [ ]:
def get_star_weights_from_2d_logic(df_stars, obj_cols, rm, rp, mapping_params, prev_target_pos):
    """
    df_stars: Index=star_idx, Cols=['boom_val', 'crash_val']
    mapping_params (5 total):
    - [0]: Sigma (Gaussian width/blur)
    - [1]: Scale (Mapping R to path length)
    - [2]: Offset (Alignment on path)
    - [3]: Moderation Strength (How much RP affects RM)
    - [4]: Hysteresis Constant (Eta - 'stickiness' on the way down)
    """
    # 1. Path Calculation: Linearize the 2D Pareto Frontier
    objs = df_stars[obj_cols].values
    # Normalize objectives to (0,1) so distances are objective-neutral
    objs_norm = (objs - objs.min(axis=0)) / (objs.max(axis=0) - objs.min(axis=0) + 1e-9)
    
    diffs = np.diff(objs_norm, axis=0)
    step_dists = np.sqrt((diffs**2).sum(axis=1))
    path_coords = np.concatenate(([0], np.cumsum(step_dists)))
    
    # 2. Calculate Moderated Target Position
    # persistence (rp) scales the magnitude (rm)
    moderated_rm = rm * (1 + mapping_params[3] * rp)
    current_raw_pos = (moderated_rm * mapping_params[1]) + mapping_params[2]
    
    # 3. Apply Asymmetric Hysteresis
    # Follow risk UP instantly; require 'eta' drop to move back DOWN toward Boom
    eta = mapping_params[4]
    
    if current_raw_pos >= prev_target_pos:
        # Increasing risk: immediate response
        final_target_pos = current_raw_pos
    else:
        # Decreasing risk: check if drop exceeds 'stickiness' threshold
        if (prev_target_pos - current_raw_pos) < eta:
            final_target_pos = prev_target_pos
        else:
            final_target_pos = current_raw_pos
            
    # 4. Generate Normalized Gaussian Weights
    sigma = max(0.01, mapping_params[0])
    dist_sq = (path_coords - final_target_pos)**2
    weights = np.exp(-dist_sq / (2 * sigma**2))
    
    weights_series = pd.Series(
        weights / (weights.sum() + 1e-9), 
        index=df_stars.index
    )
    
    return weights_series, final_target_pos

In [ ]:
import numpy as np

# Parameter Names for tracking and logging
parameter_names = [
    # --- 12 Temporal Parameters ---
    "alpha_0", "alpha_1", "alpha_2", "alpha_3",       # EMAs for the 4 signals
    "mag_w0", "mag_w1", "mag_w2", "mag_w3",           # Magnitude weights (RM)
    "pers_w0", "pers_w1", "pers_w2", "pers_w3",       # Persistence weights (RP)
    
    # --- 5 Spatial/Logic Parameters ---
    "sigma",                                          # Gaussian blur (Specialist vs Generalist)
    "scale",                                          # Signal-to-Path scaling
    "offset",                                         # Path alignment
    "moderation",                                     # RP influence on RM
    "hysteresis_eta"                                  # Asymmetric lag (turnover control)
]

# Lower Bounds (xl)
xl = np.array([
    # Temporal: Alphas (min 0.01 for slow moving averages)
    0.01, 0.01, 0.01, 0.01,
    # Temporal: Magnitude Weights (allows inverse correlation)
    -2.0, -2.0, -2.0, -2.0,
    # Temporal: Persistence Weights
    -1.0, -1.0, -1.0, -1.0,
    
    # Spatial: Sigma (0.001 allows "Hard Switching" to one star)
    0.001,
    # Spatial: Scale (Minimum stretch)
    0.1,
    # Spatial: Offset
    -2.0,
    # Spatial: Moderation
    -1.0,
    # Spatial: Hysteresis (0.0 means react instantly to all moves)
    0.0
])

# Upper Bounds (xu)
xu = np.array([
    # Temporal: Alphas (0.95 is nearly raw daily signal)
    0.95, 0.95, 0.95, 0.95,
    # Temporal: Magnitude Weights
    2.0, 2.0, 2.0, 2.0,
    # Temporal: Persistence Weights
    1.0, 1.0, 1.0, 1.0,
    
    # Spatial: Sigma (1.5 allows wide blending across the frontier)
    1.5,
    # Spatial: Scale
    5.0,
    # Spatial: Offset
    2.0,
    # Spatial: Moderation
    1.0,
    # Spatial: Hysteresis (0.5 requires a 50% drop to rotate back)
    0.5
])